# Interrupts

Interrupts in LangGraph let us pause graph execution at any point, persist the entire graph state, and wait for external input (such as a human decision) before continuing.

When `interrupt(value)` is called inside a node, the graph stops immediately and saves its exact state via the persistence layer (checkpointer).

### Pause using `interrupt`

The interrupt function pauses graph execution and returns a value to the caller. When we call interrupt within a node, LangGraph saves the current graph state and waits for resume execution with input.

Require:
- Checkpointer
- Thread ID
- Call `interrupt()`

```python
from langgraph.types import interrupt

def approval_node(state: State):
    # Pause and ask for approval
    approved = interrupt("Do you approve this action?")

    # When you resume, Command(resume=...) returns that value here
    return {"approved": approved}

What happens:
1. Graph execution gets suspended at the exact point where `interrupt` is called
2. State is saved using the checkpointer so execution can be resumed later, In production, this should be a persistent checkpointer (e.g. backed by a database)
3. Value is returned to the caller under __interrupt__; it can be any JSON-serializable value (string, object, array, etc.)
4. Graph waits indefinitely until you resume execution with a response
5. Response is passed back into the node when you resume, becoming the return value of the `interrupt()` call

### Resuming `interrupts`

After an interrupt pauses execution, you resume the graph by invoking it again with a Command that contains the resume value.

```python
from langgraph.types import Command

# Initial run - hits the interrupt and pauses
# thread_id is the persistent pointer (stores a stable ID in production)
config = {"configurable": {"thread_id": "thread-1"}}
result = graph.invoke({"input": "data"}, config=config)

# Check what was interrupted
# __interrupt__ contains the payload that was passed to interrupt()
print(result["__interrupt__"])
# > [Interrupt(value='Do you approve this action?')]

# Resume with the human's response
# The resume payload becomes the return value of interrupt() inside the node
graph.invoke(Command(resume=True), config=config)

Key about resuming:
- You must use the same `thread_id` when resuming that was used when the interrupt occurred
- The value passed to `Command(resume=...)` becomes the return value of the interrupt call
- The node restarts from the beginning of the node where the interrupt was called when resumed, so any code before the interrupt runs again
- You can pass any JSON-serializable value as the resume value


### Common patterns
The key thing that interrupts unlock is the ability to pause execution and wait for external input. This is useful for a variety of use cases, including:

- **Approval workflows**: Pause before executing critical actions (API calls, database changes, financial transactions)
- **Review and edit**: Let humans review and modify LLM outputs or tool calls before continuing
- **Interrupting tool calls**: Pause before executing tool calls to review and edit the tool call before execution
- **Validating human input**: Pause before proceeding to the next step to validate human input
